# 0. Dependencies

In [1]:
# @title Imports

import pandas as pd
import scipy.sparse as sp

!pip install igraph
!pip install -U kaleido
!pip install networkx==2.7
!sudo pip install python-igraph
import igraph
from typing import Mapping, Union, Optional
from pathlib import Path

import networkx as nx
import numpy as np
import argparse

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

import os
import pickle
from tqdm.notebook import tqdm
from tqdm import trange

from __future__ import print_function, division

from google.colab import drive
drive.mount('/content/drive')

from IPython.display import clear_output
clear_output(wait=False)

In [56]:
# @title Graph Utility Functions
def get_nodes(df):
  df1 = df[['Dependent', 'D_type', 'Speaker1']].copy()
  df1.columns = ['node', 'type', 'speaker']

  df2 = df[['Governor', 'G_type', 'Speaker2']].copy()
  df2.columns = ['node', 'type', 'speaker']

  nodes = pd.concat([df1, df2])
  nodes = nodes.dropna().drop_duplicates().reset_index(drop=True, allow_duplicates=False)
  return nodes


def get_edges(df):
  df1 = df[['Dependent', 'D_type', 'Speaker1','Governor', 'G_type', 'Speaker2', 'RelationType']].copy()
  df1.columns = ['node1', 'type1', 'speaker1', 'node2', 'type2', 'speaker2', 'relation']
  df1 = df1.dropna().drop_duplicates().reset_index(drop=True, allow_duplicates=False)
  return df1

def get_adj_matrix_and_labels(df,
                              relation_label={'Equivalent': 1,'Support': 2,'Attack': 3},
                              node_label={'Claim': 0, 'Premise': 1}):

  nodes = get_nodes(df)
  edges = get_edges(df)

  num_nodes = len(nodes)

  node_to_index = {(node, node_type): index for index, (node, node_type) in nodes[['node', 'type']].iterrows()}

  adj_matrix = np.zeros((num_nodes, num_nodes), dtype=int)

  edge_dict = {(row['node1'], row['type1'], row['node2'],row['type2']): relation_label.get(row['relation'], 0) \
               for _, row in edges.iterrows()}

  with tqdm(total=num_nodes) as pbar:

        for i, (dependent, type_d) in nodes[['node', 'type']].iterrows():
            for j, (governor, type_g) in nodes[['node', 'type']].iterrows():

                relation = edge_dict.get((dependent, type_d, governor, type_g), 0)
                reverse_relation = edge_dict.get((governor, type_g, dependent, type_d), 0)

                adj_matrix[i, j] = relation
                adj_matrix[j, i] = reverse_relation

            pbar.update(1)

  node2labels = nodes['type'].map(node_label).tolist()

  node2speaker = nodes['speaker'].tolist()

  node2text = nodes['node'].tolist()

  return adj_matrix, node2labels, node2text, node2speaker

def plot_graph(adj, node_colors, node_text, node_speaker, colors_legend=['Claim', 'Premise']):
    N = adj.shape[0]

    G = igraph.Graph.Weighted_Adjacency(adj, mode='directed')

    layt = G.layout('fr_3d')

    # Node coordinates
    Xn = [layt[k][0] for k in range(N)]
    Yn = [layt[k][1] for k in range(N)]
    Zn = [layt[k][2] for k in range(N)]

    # Edge coordinates
    Xe = []
    Ye = []
    Ze = []

    edgeA, edgeB = np.where(adj >= 1)

    for x,y in zip(edgeA, edgeB):
      Xe+=[layt[x][0], layt[y][0], None]
      Ye+=[layt[x][1], layt[y][1], None]
      Ze+=[layt[x][2], layt[y][2], None]

    edge_color_map = {1: 'blue', 2: 'green', 3: 'red'}

    edge_colors = []
    for weight in adj[adj>0].ravel():
      edge_colors+=[edge_color_map[weight],edge_color_map[weight],edge_color_map[weight]]

    trace_edges = go.Scatter3d(
        x=Xe,
        y=Ye,
        z=Ze,
        name='relations',
        mode='lines',
        line=dict(
            color=edge_colors,  # Use the list of edge colors
            width=2
        ),
        hoverinfo='none',
        legendgroup='Edges'
    )

    markers_dict = {0: 'circle', 1: 'square', 2: 'cross', 3: 'diamond', 4: 'x'}


    speaker_to_index = {speaker: index for index, speaker in enumerate(np.unique(node_speaker))}
    node_idx_speaker = [speaker_to_index[speaker] for speaker in node_speaker]

    trace_nodes = go.Scatter3d(
        x=Xn,
        y=Yn,
        z=Zn,
        mode='markers',
        name="Nodes",
        marker=dict(
            symbol=[markers_dict[speaker] for speaker in node_idx_speaker],
            size=6,
            color=node_colors,
            colorscale='Viridis',
            line=dict(color='rgb(50,50,50)', width=0.5),
        ),
        hoverlabel=dict(
          bgcolor='white',
          bordercolor='black',
          font=dict(size=10)  # Adjust the text size (e.g., size=16)
        ),
        text=node_text,
        hoverinfo='text',
        legendgroup='Nodes',
        showlegend=True
    )

    unique_speakers = np.unique(node_speaker)
    num_unique_speakers = len(unique_speakers)

    shapes = [markers_dict[i] for i in range(num_unique_speakers)]
    title = f'Debate between {", ".join(unique_speakers[:-1])}, and {unique_speakers[-1]} (With shapes: {", ".join(shapes)})'

    layout = go.Layout(
        title=title,
        width=800,
        height=800,
        showlegend=True,
        scene=dict(
            xaxis=dict(showbackground=False, showline=False, zeroline=False, showgrid=False, showticklabels=False, title=''),
            yaxis=dict(showbackground=False, showline=False, zeroline=False, showgrid=False, showticklabels=False, title=''),
            zaxis=dict(showbackground=False, showline=False, zeroline=False, showgrid=False, showticklabels=False, title='')
        ),
        legend_title=dict(text='Legend'),
        margin=dict(t=100),
        hovermode='closest',
    )

    fig = go.Figure(data=[trace_edges, trace_nodes], layout=layout)
    print("Claim: yellow, Premise: purle")
    print("Equivalent relation: blue, Support: green, Attack:red")
    return fig

# 1. Exploratory Data Analysis

In [3]:
dataset = "/content/drive/MyDrive/DORE_tesi_magistrale/dataset/final_relation_graph.csv"

In [4]:
df = pd.read_csv(dataset)
df.head()

,Year,date,Dependent,D_type,Speaker1,Governor,G_type,Speaker2,RelationType,long_date
0,1960,07 10,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,For me to have made such a statement would bee...,Claim,NIXON,Support,07-10-1960
1,1960,07 10,"Now I'm very surprised that Senator Kennedy, w...",Claim,NIXON,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,Support,07-10-1960
2,1960,07 10,"Now I'm very surprised that Senator Kennedy, w...",Premise,NIXON,Senator Kennedy also indicated with regard to ...,Claim,NIXON,Support,07-10-1960
3,1960,07 10,"I look at Cuba today, I believe that we are fo...",Claim,NIXON,We think that's pretty good progress,Claim,NIXON,Support,07-10-1960
4,1960,07 10,"I look at Cuba today, I believe that we are fo...",Premise,NIXON,a course which is difficult,Claim,NIXON,Attack,07-10-1960


In [5]:
print(f"Edges in the corpus {len(df)}")

Edges in the corpus 26230


In [6]:
nodes = get_nodes(df)
print(f"Nodes in the corpus {len(nodes)} ")

Nodes in the corpus 38667 


In [7]:
nodes.head()

,node,type,speaker
0,"As a matter of fact in his book, The Strategy ...",Claim,NIXON
1,"Now I'm very surprised that Senator Kennedy, w...",Claim,NIXON
2,"Now I'm very surprised that Senator Kennedy, w...",Premise,NIXON
3,"I look at Cuba today, I believe that we are fo...",Claim,NIXON
4,"I look at Cuba today, I believe that we are fo...",Premise,NIXON


In [8]:
edges = get_edges(df)
edges.head()

,node1,type1,speaker1,node2,type2,speaker2,relation
0,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,For me to have made such a statement would bee...,Claim,NIXON,Support
1,"Now I'm very surprised that Senator Kennedy, w...",Claim,NIXON,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,Support
2,"Now I'm very surprised that Senator Kennedy, w...",Premise,NIXON,Senator Kennedy also indicated with regard to ...,Claim,NIXON,Support
3,"I look at Cuba today, I believe that we are fo...",Claim,NIXON,We think that's pretty good progress,Claim,NIXON,Support
4,"I look at Cuba today, I believe that we are fo...",Premise,NIXON,a course which is difficult,Claim,NIXON,Attack


In [9]:
relations = pd.Series(df['RelationType']).drop_duplicates().tolist()
relations

['Support', 'Attack', 'Equivalent']

In [10]:
speeches = df['long_date'].unique().tolist()
len(speeches)

44

## Visualization

In [11]:
# generate a df per debate
deb_list = []
for date in speeches:
  deb = df[df['long_date']==str(date)].copy()
  deb = deb.drop(labels=['Year', 'date'], axis=1).reset_index(drop=True)
  deb_list.append(deb)

In [62]:
deb_list[10].head()

,Dependent,D_type,Speaker1,Governor,G_type,Speaker2,RelationType,long_date
0,This is the greatest democracy in the world,Premise,KEMP,This is a democracy in which we have the frees...,Claim,KEMP,Support,09-10-1996
1,I really got only two differences with Bill Cl...,Claim,KEMP,"Our foreign policy is ambivalent, confusing, i...",Claim,KEMP,Support,09-10-1996
2,"Our foreign policy is ambivalent, confusing, i...",Premise,KEMP,we have learned over the years that weakness i...,Claim,KEMP,Support,09-10-1996
3,Bob Dole and I believe we can do a lot better,Claim,KEMP,domestic economy is not doing what it can do,Claim,KEMP,Support,09-10-1996
4,Bob Dole and I believe we can do a lot better,Premise,KEMP,It is about the potential of the American peop...,Claim,KEMP,Support,09-10-1996


In [63]:
adj, node_labels, node_text, node_speaker = get_adj_matrix_and_labels(deb_list[10])

  0%|          | 0/784 [00:00<?, ?it/s]

In [64]:
fig  = plot_graph(adj, node_labels, node_text, node_speaker)
plt.tight_layout()
fig.show()

Claim: yellow, Premise: purle
Equivalent relation: blue, Support: green, Attack:red


<Figure size 640x480 with 0 Axes>

In [65]:
fig.write_image("09-10-1996_debate_graph.svg")